# 3_neuron_ur_all_analysis — single-neuron PSTH grids for all UR neurons

Reproduces **Extended Fig. 5a (CUE) and 5b (CUEP)**: single-neuron PSTHs for
**all** `CUE_UR_NEURONS` and `CUEP_UR_NEURONS` (9 neurons each), as one
9-neuron x 4-panel grid per task prefix:

- columns 0-1: PD-chosen trials (`~intoStage2`), grouped by coherence and by RT quintile;
- columns 2-3: UR-chosen trials (`intoStage2`), grouped by 4-level coherence and by RT quintile.

Each row is one neuron; traces are aligned to stimulus onset (`stim_idx`) via
`utils_neuron_analysis.getAlignedMeanFr`, with SEM shading. One time bin is one
environment step (dt = 50 ms, from the param file in `saved_params/`).

**Prerequisite:** run `1_generate_data.ipynb` first — this notebook reads the simulated
data from `data/` (`CUE_part*.pkl` and `CUEP_part*.pkl`). The `CUEP_part*_never_ur.pkl`
never-UR control files are explicitly excluded. Figures display inline only; nothing is
saved to disk.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
import warnings

from utils_type_neuron import CUE_UR_NEURONS, CUEP_UR_NEURONS
from utils_neuron_analysis import getAlignedMeanFr, COPPER_TRUNCATED, FixedOrderFormatter

# Globally ignore RuntimeWarning (e.g., mean of empty slice)
warnings.simplefilter("ignore", category=RuntimeWarning)

df_pkl_kwargs = {"compression": {"method": "gzip", "compresslevel": 4, "mtime": 1}}


In [ ]:
from matplotlib import colormaps
from matplotlib.colors import LinearSegmentedColormap

# Truncated reversed 'Blues' (light half only) for PD-trial traces
blues_r_truncated = LinearSegmentedColormap.from_list(
    'blues_r_truncated', colormaps['Blues_r'](np.linspace(0.0, 0.5, 256))
)

In [ ]:
def prepare_single_neuron_df(df_base, neuron_idx):
    """Build fr_{neuron_idx}, apply afterchcutoff, pad with NaNs.
    Returns dfu (intoStage2), dfp (~intoStage2), and frn_col.
    """
    df_local = df_base.copy()
    frn_col = f'fr_{neuron_idx}'

    def afterchcutoff(x):
        if x['ch_idx'] > 0:
            return x['fr'][:x['ch_idx'], neuron_idx]
        return x['fr'][:, neuron_idx]

    df_local[frn_col] = df_local.apply(afterchcutoff, axis=1)
    max_length = max(df_local[frn_col].apply(lambda x: x.shape[0]))
    df_local[frn_col] = df_local[frn_col].apply(
        lambda x: np.r_[x, np.ones(max_length - len(x)) * np.nan]
    )

    dfu_local = df_local.query('intoStage2')
    dfp_local = df_local.query('~intoStage2')
    return dfu_local, dfp_local, frn_col

In [ ]:
def plot_grouped_psth_grid(data, this_ax, legend_title, cmap, term, xlim, noise='sem'):
    """Plot one PSTH panel: one trace per group in `data`, with SEM shading.

    Returns
    -------
    handles, labels : list, list
        Line handles and their labels for building legends outside.
    """
    noise_mult = 1.96 if noise == 'ci95' else 1.0
    group_keys = sorted(data.keys(), key=lambda x: (str(x) if isinstance(x, int) else x))
    if isinstance(cmap, str):
        color_map = plt.get_cmap(cmap)
    else:
        color_map = cmap
    # Spread colors evenly across the full colormap so
    # each group (line) is clearly distinguishable
    colors = color_map(np.linspace(0, 1, len(group_keys)))
    # Color/legend orderings below match the paper figures
    if term=='dircohc':
        colors[-1], colors[-2] = colors[-2].copy(), colors[-1].copy()
    elif term=='cohq':
        colors = colors[::-1].copy()  # Highest coherence darkest
    elif term=='rtgc':
        colors[0], colors[1] = colors[1].copy(), colors[0].copy()

    handles = []
    labels = []

    for idx, (group_val, d) in enumerate(sorted(data.items(), key=lambda x: (str(x[0]) if isinstance(x[0], int) else x[0]))):
        time_axis = d['time_axis']
        mean_fr = d['mean_fr']
        sem_fr = d['sem_fr']

        # Cap the plotted trace at 100 bins (100 x 50 ms = 5 s) post-alignment
        plotcutoff = 100
        plot_idx = min(plotcutoff, len(time_axis))
        time_axis_plot = time_axis[:plot_idx]
        mean_fr_plot = mean_fr[:plot_idx]
        sem_fr_plot = sem_fr[:plot_idx]

        # Restrict to common xlim
        if xlim is not None:
            start_idx = np.searchsorted(time_axis_plot, xlim[0], side="left")
            end_idx = np.searchsorted(time_axis_plot, xlim[1], side="right")
            time_axis_plot = time_axis_plot[start_idx:end_idx]
            mean_fr_plot = mean_fr_plot[start_idx:end_idx]
            sem_fr_plot = sem_fr_plot[start_idx:end_idx]

        # 'cohq' group labels carry an ordinal prefix for sorting
        # ('1 Lowest' .. '4 Highest'); strip it for display
        label = (str(group_val)[2:] if term=='cohq' else f'{group_val}') if legend_title == "Coherence" else f'{group_val}'
        line_color = colors[idx]

        line, = this_ax.plot(time_axis_plot, mean_fr_plot, color=line_color, label=label)
        if noise is not None:
            this_ax.fill_between(time_axis_plot, mean_fr_plot - noise_mult * sem_fr_plot,
                                 mean_fr_plot + noise_mult * sem_fr_plot,
                                 color=line_color, alpha=0.2, linewidth=0)
        handles.append(line)
        labels.append(label)

    this_ax.axvline(x=0, color='k', linestyle='--', linewidth=1, alpha=0.7)
    if xlim is not None:
        this_ax.set_xlim(left=xlim[0], right=xlim[1])

    if term=='dircohc':
        handles[-1], handles[-2] = handles[-2], handles[-1]
        labels[-1], labels[-2] = labels[-2], labels[-1]
    elif term=='cohq':
        handles.reverse()
        labels.reverse()
    elif term=='rtgc':
        handles[0], handles[1] = handles[1], handles[0]
        labels[0], labels[1] = labels[1], labels[0]

    return handles, labels

In [ ]:
# Grid plot over all UR neurons: one (n_neurons x 4) PSTH grid per task prefix (CUE and CUEP)
for cue_prefix in ['CUE', 'CUEP']:
    neuron_list = CUE_UR_NEURONS if cue_prefix == 'CUE' else CUEP_UR_NEURONS
    print(f"cue_prefix = {cue_prefix}, number of UR neurons = {len(neuron_list)}")

    # Load this prefix's REGULAR data parts only; a bare CUEP_part*.pkl glob would
    # silently also match the CUEP_part*_never_ur.pkl never-UR control files.
    paths = [p for p in sorted(Path('data').glob(f'{cue_prefix}_part*.pkl')) if '_never_ur' not in p.name]
    dfs = [pd.read_pickle(p, **df_pkl_kwargs) for p in paths]
    df = pd.concat(dfs, ignore_index=True)

    COH = np.unique(df['coh'])
    assert len(COH) == 8
    # 2-level coherence grouping for PD trials
    cohc_dict = {
        2: 'Low', 4: 'Low', 6: 'Low', 8: 'Low',
        13: 'High', 26: 'High', 52: 'High', 78: 'High'
    }
    df['cohc'] = df['coh'].apply(lambda x: cohc_dict[x])
    sign_dict = {-1: 'Left', 1: 'Right'}
    df['dircohc'] = df['dircoh'].apply(
        lambda x: sign_dict[np.sign(x)] + ' ' + cohc_dict[np.abs(x)]
    )

    # 4-level coherence grouping for UR trials (ordinal prefix fixes sort order)
    COH4_DICT = {2: '1 Lowest', 4: '1 Lowest', 6: '2 Lower', 8: '2 Lower',
                 13: '3 Higher', 26: '3 Higher', 52: '4 Highest', 78: '4 Highest'}

    nrows = len(neuron_list)
    ncols = 4  # dfp: (cohc, rtg), dfu: (cohq, rtg)

    fig, ax = plt.subplots(nrows=nrows, ncols=ncols, sharex=True, sharey='row', figsize=(8, 13.5), dpi=100)

    xlim = (-500, 2000)

    for row, neuron_idx in enumerate(neuron_list):
        print(f"\n=== Neuron index in list: {row}, neuron_id: {neuron_idx} ===")

        # Build single-neuron firing rate column and stage-2 splits
        dfu, dfp, frn_col = prepare_single_neuron_df(df, neuron_idx)

        # --- dfp (not intoStage2) ---
        # Column 0: coherence grouping
        data_dfp_cohc = getAlignedMeanFr(
            dfp,
            frn_col,
            align_col='stim_idx',
            term='dircohc'
        )
        handles_dfp_cohc, labels_dfp_cohc = plot_grouped_psth_grid(
            data_dfp_cohc, ax[row, 0], legend_title='Coherence', cmap=blues_r_truncated, term='dircohc', xlim=xlim
        )
        if row == 0:
            ax[row, 0].legend(handles_dfp_cohc, labels_dfp_cohc, loc='upper right', frameon=False, fontsize=8)

        # Column 1: RT Groups grouping
        dfp_cfg = dfp.copy()
        dfp_cfg['rtg'] = pd.qcut(dfp_cfg['rt'], q=5, labels=False, duplicates='drop')
        rtg_dict = {0: 'Shortest', 1: 'Shorter', 2: 'Normal', 3: 'Longer', 4: 'Longest'}
        dfp_cfg['rtgc'] = dfp_cfg['rtg'].apply(lambda x: rtg_dict[x])
        data_dfp_rtg = getAlignedMeanFr(
            dfp_cfg,
            frn_col,
            align_col='stim_idx',
            term='rtgc'
        )
        handles_dfp_rtg, labels_dfp_rtg = plot_grouped_psth_grid(
            data_dfp_rtg, ax[row, 1], legend_title='RT Groups', cmap=blues_r_truncated, term='rtgc', xlim=xlim
        )
        if row == 0:
            ax[row, 1].legend(handles_dfp_rtg, labels_dfp_rtg, loc='upper right', frameon=False, fontsize=8)

        # --- dfu (intoStage2) ---
        # Column 2: coherence grouping
        dfu_coh = dfu.copy()
        dfu_coh['cohq'] = dfu_coh['coh'].apply(lambda x: COH4_DICT[x])
        data_dfu_cohc = getAlignedMeanFr(
            dfu_coh,
            frn_col,
            align_col='stim_idx',
            term='cohq'
        )
        handles_dfu_cohc, labels_dfu_cohc = plot_grouped_psth_grid(
            data_dfu_cohc, ax[row, 2], legend_title='Coherence', cmap=COPPER_TRUNCATED, term='cohq', xlim=xlim
        )
        if row == 0:
            ax[row, 2].legend(handles_dfu_cohc, labels_dfu_cohc, loc='upper right', frameon=False, fontsize=8)

        # Column 3: RT Groups grouping
        dfu_cfg = dfu.copy()
        dfu_cfg['rtg'] = pd.qcut(dfu_cfg['rt'], q=5, labels=False, duplicates='drop')
        dfu_cfg['rtgc'] = dfu_cfg['rtg'].apply(lambda x: rtg_dict[x])
        data_dfu_rtg = getAlignedMeanFr(
            dfu_cfg,
            frn_col,
            align_col='stim_idx',
            term='rtgc'
        )
        handles_dfu_rtg, labels_dfu_rtg = plot_grouped_psth_grid(
            data_dfu_rtg, ax[row, 3], legend_title='RT Groups', cmap=COPPER_TRUNCATED, term='rtgc', xlim=xlim
        )
        if row == 0:
            ax[row, 3].legend(handles_dfu_rtg, labels_dfu_rtg, loc='upper right', frameon=False, fontsize=8)

        # Labels: only first column gets y-label (just neuron number); only top row gets titles
        ax[row, 0].set_ylabel(f'# {neuron_idx}')

    # Column titles on top row
    ax[0, 0].set_title('PD chosen\nCoherence')
    ax[0, 1].set_title('PD chosen\nRT Groups')
    ax[0, 2].set_title('UR chosen\nCoherence')
    ax[0, 3].set_title('UR chosen\nRT Groups')

    # Remove right and top spines for all subplots
    for r in range(nrows):
        for c in range(ncols):
            ax[r, c].spines['right'].set_visible(False)
            ax[r, c].spines['top'].set_visible(False)

    # Figure-level labels
    fig.supxlabel("Time (ms)")
    fig.supylabel("Firing Rate (a.u.)")
    # Apply scientific notation to y-axis
    for r in range(nrows):
        formatter = FixedOrderFormatter(order_of_magnitude=-2, useMathText=True)
        ax[r, 0].yaxis.set_major_formatter(formatter)

    fig.tight_layout(rect=[0, 0, 1, 0.97])
    plt.show()
